# Crawl VietnamWorks: hiện tại và lịch sử

Chạy lần lượt từ trên xuống. Mặc định `MODE = "live"`, xử lý tối đa 50 URL chưa xử lý trong mỗi lượt. Đặt `CRAWL_LIMIT = None` để crawl hết; chạy lại cell crawl để tiếp tục.

Dùng `curl_cffi`, một luồng, giãn request và lưu checkpoint. URL hiện tại lấy từ sitemap chính thức; trang mới có dữ liệu Next.js nhúng trong HTML, không cần trình duyệt hay chạy JavaScript.

Chọn `MODE = "history"` hoặc `"both"` để lấy snapshot Wayback. Dữ liệu lịch sử phụ thuộc độ phủ và khả năng truy cập của Archive.org; giao diện cũ chỉ nhận khi có JobPosting JSON-LD hoặc dữ liệu nhúng phù hợp.


## 1. Môi trường

Chọn Python kernel của dự án. Nếu thiếu thư viện, bỏ dấu `#` và chạy cell cài đặt một lần. Sau khi cập nhật code crawler, restart kernel trước khi chạy lại notebook.


In [ ]:
# %pip install pandas beautifulsoup4 curl_cffi tqdm


In [ ]:
import sys
from pathlib import Path

# Tự tìm thư mục dự án khi mở notebook từ root hoặc notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/crawler/vietnamworks_crawler.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong thư mục project_intro_ds hoặc notebooks của dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.crawler import VietnamWorksCrawler

print("Project:", PROJECT_ROOT)


## 2. Cấu hình chung

`HISTORY_START_YEAR` và `HISTORY_END_YEAR` lọc **năm chụp Wayback**, không phải năm đăng tin. Mặc định 2022–2025. Danh sách lịch sử đã lưu được bổ sung khi mở rộng khoảng năm; muốn bộ dữ liệu riêng, đổi đường dẫn file lịch sử.

Giữ `RESUME_JOBS = True` để chạy tiếp. `False` ghi đè CSV kết quả của nguồn được chọn. `REFRESH_LIVE_URLS` chỉ gom lại sitemap hiện tại. Gom lịch sử luôn chạy tiếp theo checkpoint CDX.


In [ ]:
MODE = "live"                 # "live", "history", "both"
CRAWL_LIMIT = 50              # Giới hạn URL mỗi nguồn, mỗi lượt; None = tất cả
RESUME_JOBS = True
SAVE_EVERY = 50
REFRESH_LIVE_URLS = False

HISTORY_START_YEAR = 2022
HISTORY_END_YEAR = 2025
LIVE_REQUEST = dict(min_delay=3.0, max_delay=6.0, timeout=30)
HISTORY_REQUEST = dict(min_delay=5.0, max_delay=10.0, timeout=60)

RAW_DIR = PROJECT_ROOT / "data/raw"
URL_FILE = RAW_DIR / "vietnamworks_it_job_urls.csv"
JOB_FILE = RAW_DIR / "vietnamworks_it_jobs.csv"
HISTORY_URL_FILE = RAW_DIR / "vietnamworks_history_urls.csv"
HISTORY_JOB_FILE = RAW_DIR / "vietnamworks_history_jobs.csv"

if MODE not in {"live", "history", "both"}:
    raise ValueError("MODE phải là live, history hoặc both")
RUN_LIVE = MODE in {"live", "both"}
RUN_HISTORY = MODE in {"history", "both"}
print(f"Chế độ: {MODE}; giới hạn mỗi nguồn: {CRAWL_LIMIT}")
print(f"Khoảng năm chụp lịch sử: {HISTORY_START_YEAR}–{HISTORY_END_YEAR}")
print("Thư mục kết quả:", RAW_DIR)


## 3. Gom URL hiện tại

Cell tự bỏ qua nếu chọn `history`. Dùng lại danh sách URL hiện có, trừ khi bật `REFRESH_LIVE_URLS`.


In [ ]:
if RUN_LIVE:
    if URL_FILE.exists() and not REFRESH_LIVE_URLS:
        live_urls = pd.read_csv(URL_FILE)
    else:
        with VietnamWorksCrawler(**LIVE_REQUEST) as crawler:
            live_urls = crawler.collect_urls_from_sitemaps(save_path=str(URL_FILE))
    if "url" not in live_urls.columns:
        raise ValueError("File URL hiện tại thiếu cột url")
    print(f"Hiện tại: {len(live_urls):,} URL")
    display(live_urls.head())
else:
    print("Bỏ qua nguồn hiện tại theo MODE.")


## 4. Gom snapshot lịch sử

Cell tự bỏ qua khi `MODE = "live"`. CDX truy vấn VietnamWorks, lọc URL tin `-ID-jv` theo năm chụp, chọn một snapshot mỗi URL/năm và lưu cursor sau mỗi trang.

Wayback chỉ có các trang từng được lưu; bộ lọc slug IT/Data có thể bỏ sót tiêu đề chung chung. Khi timeout hoặc 403/429, có thể chạy lại cell sau để tiếp tục. Chưa chạy crawl lịch sử thực tế trong lần bổ sung này.


In [ ]:
if RUN_HISTORY:
    with VietnamWorksCrawler(**HISTORY_REQUEST) as crawler:
        history_urls = crawler.collect_historical_urls(
            start_year=HISTORY_START_YEAR,
            end_year=HISTORY_END_YEAR,
            save_path=str(HISTORY_URL_FILE),
            resume=True,
        )
    print(f"Lịch sử: {len(history_urls):,} snapshot đã gom")
    display(history_urls.head())
else:
    print("Bỏ qua nguồn lịch sử theo MODE.")


## 5. Crawl chi tiết hiện tại

Chạy lại cell crawl của từng nguồn để xử lý lượt tiếp theo. CSV ghi ngay sau mỗi tin thành công; state lưu định kỳ và khi kết thúc. Có thể Interrupt kernel để dừng. Khi 403/429, crawler dừng và giữ URL chưa xử lý.


In [ ]:
if RUN_LIVE:
    if not URL_FILE.exists():
        raise FileNotFoundError("Chạy cell gom URL hiện tại trước")
    with VietnamWorksCrawler(**LIVE_REQUEST) as crawler:
        live_jobs = crawler.crawl_jobs_from_url_file(
            url_file_path=str(URL_FILE), save_path=str(JOB_FILE),
            limit=CRAWL_LIMIT, save_every=SAVE_EVERY, resume=RESUME_JOBS,
        )
    print(f"Hiện tại: đã lưu {len(live_jobs):,} tin")
    display(live_jobs.tail())
else:
    print("Bỏ qua nguồn hiện tại theo MODE.")


## 6. Crawl chi tiết lịch sử

Tải HTML snapshot đã chọn. Nếu snapshot chuyển hướng hoặc không đọc được, crawler giữ lại để thử sau; không tự lấy trang live thay thế. Checkpoint nhận diện bằng URL snapshot, cho phép cùng một tin xuất hiện ở các năm khác nhau.


In [ ]:
if RUN_HISTORY:
    if not HISTORY_URL_FILE.exists():
        raise FileNotFoundError("Chạy cell gom snapshot lịch sử trước")
    with VietnamWorksCrawler(**HISTORY_REQUEST) as crawler:
        history_jobs = crawler.crawl_historical_jobs(
            url_file_path=str(HISTORY_URL_FILE), save_path=str(HISTORY_JOB_FILE),
            limit=CRAWL_LIMIT, save_every=SAVE_EVERY, resume=RESUME_JOBS,
        )
    print(f"Lịch sử: đã lưu {len(history_jobs):,} bản ghi snapshot")
    display(history_jobs.tail())
else:
    print("Bỏ qua nguồn lịch sử theo MODE.")


## 7. Đọc kết quả đã lưu

Cell này không gửi request. Cùng 10 cột với TopCV/CareerViet; lịch sử có thêm `archive_timestamp`, `archive_url`.

Trang Next.js: `posted_date` lấy từ `approvedOn` (ngày duyệt đăng); `deadline_date` từ `expiredOn`. Không lấy ngày đẩy tin `onlineOn`, ngày đồng bộ, sitemap `lastmod` hay ngày snapshot làm ngày đăng. Trang JSON-LD cũ lấy `datePosted`/`validThrough`. Thiếu ngày thì để trống.

Khi thống kê theo năm, khử trùng theo ID tin/URL và dùng `posted_date`; năm chụp Wayback không phải năm đăng. Không thể suy ra toàn bộ thị trường tuyển dụng từ độ phủ sitemap/Wayback.


In [ ]:
for label, enabled, path in (
    ("Hiện tại", RUN_LIVE, JOB_FILE),
    ("Lịch sử", RUN_HISTORY, HISTORY_JOB_FILE),
):
    if not enabled:
        continue
    if not path.exists():
        print(f"{label}: chưa có file kết quả")
        continue
    saved = pd.read_csv(path, keep_default_na=False)
    print(f"{label}: {len(saved):,} bản ghi — {path}")
    display(saved.head(10))
